In [67]:
import matplotlib
import os
import sys
sys.path.append('..')
from er_simulator.wc_model import WCTaskSim
from er_simulator.functions import resample_signal
from er_simulator.synaptic_weights_matrices import normalize, generate_synaptic_weights_matrices
from er_simulator.read_utils import generate_sw_matrices_from_mat
from er_simulator.load_wc_params import load_wc_params
from er_simulator import functions
from er_simulator.boldIntegration import BWBoldModel
import numpy as np
from tqdm import tqdm 
from tqdm import trange
import matplotlib.pyplot as plt
from scipy import signal, stats, io
from scipy.io import savemat
from er_simulator.read_utils import get_project_root


matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 10
plt.rcParams['image.cmap'] = 'plasma'
np.set_printoptions(suppress=True)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
N = 100 # Sample size
microtime = 2/16
design = "07_EVENT_[2s_TR]_[100ms_DUR]_[6s_ISI]_[100_TRIALS]_[COACT]";
HRF_type = "[FixedHRF]"; # [VarHRF]

results_folder = f'{design}_{HRF_type}'
mat_path = os.path.join('..', '02_designs', f'{design}.mat')
config_file = os.path.join('..', '01_configs', f'{results_folder}.yaml')

In [ ]:
for i in trange(N):
    
    # Create subject folder
    os.makedirs(os.path.join('..', '03_results', results_folder, fr'{i+1:03d}_Subject'), exist_ok=True)
    
    # Load simulation parameters from config file
    wc_sim = WCTaskSim.from_config(config_file)

    # Save HRF parameters
    wc_sim.boldModel.save_imp_with_params(os.path.join(fr'..', 
                                                       '03_results',
                                                       results_folder, 
                                                       fr'{i+1:03d}_Subject',
                                                       'HRF_params.mat'), s_type='mat')

    # Generate task time-series in 5 ms resolution
    wc_sim.generate_full_series(compute_bold=True)
    output_task = wc_sim.output.copy()

    # Downsample task BOLD to Micro-Time resolution = TR/16  
    task_BOLD_oscill_MT,_ = resample_signal(output_task['mtime'], output_task["BOLD"], wc_sim.mTime, microtime)

    # Generate co-activations in 5 ms resolution
    time_coact,_,task_BOLD_coact = wc_sim.generate_coactivation_by_mat(wc_sim.mat_path, dt=None, normalize_constant=1)

    # Downsample co-activations to Micro-Time resolution = TR/16  
    task_BOLD_coact_MT,_ =  resample_signal(time_coact, task_BOLD_coact,  wc_sim.mTime, microtime)
    
    # Generate rest time-series in 5 ms resolution
    output_rest = wc_sim.generate_rest_series(compute_bold=True)

    # Downsample rest BOLD to Micro-Time resolution = TR/16
    rest_BOLD_oscill_MT,_ = resample_signal(output_rest['mtime'], output_rest["BOLD"], wc_sim.mTime, microtime)

    # Save time-series
    savemat(os.path.join(fr'..',
                         '03_results',
                         results_folder,
                         fr'{i+1:03d}_Subject',
                         'time_series.mat'),
                          {'task_syn_act_5ms':    output_task['syn_act'],
                           'task_BOLD_oscill_MT': task_BOLD_oscill_MT,
                           'BOLD_coact_MT':       task_BOLD_coact_MT,
                           'rest_syn_act_5ms':    output_rest['syn_act'],
                           'rest_BOLD_oscill_MT': rest_BOLD_oscill_MT}) 
    
    # Clear
    del output_task, task_BOLD_oscill_MT, task_BOLD_coact_MT, output_rest, wc_sim 
    

  1%|▊                                                                             | 1/100 [08:24<13:52:38, 504.63s/it]